# Chat with PDFs and Other Files

Grok can answer questions about files attached to a Responses API request. This notebook shows the two common paths: reference a publicly reachable PDF by URL, or upload a private file first and then reference its `file_id`.

## Table of Contents
- [Code Setup](#code-setup)
- [Reference a Public PDF](#reference-a-public-pdf)
- [Upload a Private File](#upload-a-private-file)
- [Attach Multiple Files](#attach-multiple-files)
- [Cleanup Notes](#cleanup-notes)

## Code Setup

In [1]:
%pip install --quiet openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


> **Note:** Make sure to export an env var named `XAI_API_KEY` or set it in a `.env` file at the root of this repo if you want to run the live API calls. Head over to the [xAI Console](https://console.x.ai/) to obtain an API key if you don't have one already.

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

XAI_API_KEY = os.getenv("XAI_API_KEY")
RUN_LIVE_API = bool(XAI_API_KEY and XAI_API_KEY != "your_xai_api_key")

if not RUN_LIVE_API:
    print("Set XAI_API_KEY in your environment or .env file to run the live API examples.")

Set XAI_API_KEY in your environment or .env file to run the live API examples.


The file attachment shape follows the [xAI Chat with Files guide](https://docs.x.ai/developers/model-capabilities/files/chat-with-files): public files use `file_url`, while uploaded private files use `file_id`. Both are sent as `input_file` content parts alongside the user's text prompt.

In [3]:
BASE_URL = "https://api.x.ai/v1"
MODEL = "grok-4.3"


def make_client() -> OpenAI:
    return OpenAI(api_key=XAI_API_KEY, base_url=BASE_URL)


def output_text(response) -> str:
    """Extract the final text answer from a Responses API result."""
    for item in reversed(response.output):
        for part in getattr(item, "content", []) or []:
            if getattr(part, "type", None) == "output_text":
                return part.text
    return ""


def ask_with_files(client: OpenAI, prompt: str, file_parts: list[dict]) -> str:
    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    *file_parts,
                ],
            }
        ],
    )
    return output_text(response)

## Reference a Public PDF

If your PDF is public, skip the upload step and pass its URL directly. The model can search the attached document and answer questions about it.

In [4]:
public_pdf_url = "https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf"
prompt = "Summarize this PDF in one sentence and quote the exact title text if present."

if RUN_LIVE_API:
    client = make_client()
    answer = ask_with_files(
        client,
        prompt,
        [{"type": "input_file", "file_url": public_pdf_url}],
    )
    print(answer)
else:
    print(f"Skipping live request. Public PDF URL: {public_pdf_url}")

Skipping live request. Public PDF URL: https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf


## Upload a Private File

For PDFs or documents that are not public, upload the file first. The returned file object includes an `id` that can be referenced with `file_id` in the Responses API. Delete uploaded files after the example finishes so your account does not keep stale documents around.

In [5]:
# Replace this with your own local PDF path before running the upload example.
local_pdf_path = Path("/path/to/your/document.pdf")
uploaded_file = None

if RUN_LIVE_API and local_pdf_path.exists():
    client = make_client()
    try:
        with local_pdf_path.open("rb") as file_handle:
            uploaded_file = client.files.create(file=file_handle, purpose="assistants")

        answer = ask_with_files(
            client,
            "What are the three most important details in this document?",
            [{"type": "input_file", "file_id": uploaded_file.id}],
        )
        print(answer)
    finally:
        if uploaded_file is not None:
            client.files.delete(uploaded_file.id)
            print(f"Deleted uploaded file {uploaded_file.id}")
elif RUN_LIVE_API:
    print(f"Update local_pdf_path before running this cell: {local_pdf_path}")
else:
    print("Skipping upload example until XAI_API_KEY is set.")

Skipping upload example until XAI_API_KEY is set.


## Attach Multiple Files

You can attach more than one file in the same request. This is useful when you want Grok to compare a PDF with supporting notes, CSV data, Markdown, or another text-based document.

In [6]:
file_urls = [
    "https://docs.x.ai/assets/api-examples/documents/project-timeline.txt",
    "https://docs.x.ai/assets/api-examples/documents/project-budget.txt",
]
file_parts = [{"type": "input_file", "file_url": url} for url in file_urls]

if RUN_LIVE_API:
    client = make_client()
    answer = ask_with_files(
        client,
        "Use both files to identify the project start date and total budget.",
        file_parts,
    )
    print(answer)
else:
    print("Skipping live multi-file request. Files that would be attached:")
    for url in file_urls:
        print(f"- {url}")

Skipping live multi-file request. Files that would be attached:
- https://docs.x.ai/assets/api-examples/documents/project-timeline.txt
- https://docs.x.ai/assets/api-examples/documents/project-budget.txt


## Cleanup Notes

- Use `file_url` for public files and `file_id` for private uploads.
- Delete uploaded files when a one-off task is done.
- Prefer Collections when you need persistent semantic search across many documents rather than immediate context for a single conversation.